# Simulate Honeybee models with EnergyPlus

Runs the validated Honeybee models through EnergyPlus in resumable batches. Each model is matched
to its city's weather files, and every successful simulation produces one SQL file for the result
and graph-dataset extraction stages.

**Inputs**
- `data/interim/hbjsons_df.pkl` and the HBJSON exports in `output/hbjsons/`
- `data/interim/city_weather_assets.pkl` and the EPW/DDY files in `output/weather/`

**Requirements**
- The Python packages installed from `requirements.txt`
- OpenStudio 3.10 available as `openstudio` on `PATH`
- EnergyPlus 25.1 available on `PATH` or through `ENERGYPLUS_EXE`

**Outputs**
- Final IDF and EPJSON representations in `output/idf/` and `output/epjson/`
- One `<sample_id>.sql` per successful model in `output/simulations/`
- `data/interim/sim_results_all.pkl`, the run log containing final status, elapsed time, and messages
- Simulation folders retained only for failed models so their EnergyPlus files can be inspected

### Prepare the simulation queue

**Inputs**
- `data/interim/hbjsons_df.pkl`: validated models exported by `BEM4AI_5_BEMModels.ipynb`
- `data/interim/city_weather_assets.pkl`: weather registry written by `BEM4AI_6_Weather.ipynb`
- `data/interim/sim_results_all.pkl`: the run log, when an earlier batch exists

**Steps**
1. Attach EPW and DDY paths to every model through `city_key`.
2. Count a model as complete only when the run log records `success` and its SQL file exists.
3. Add failed, incomplete, and explicitly forced models to `PENDING_PAIRS`.

**Outputs**
- `hbjsons_df` with portable HBJSON, EPW, and DDY paths
- `PENDING_PAIRS` containing `(index, sample_id)` tuples

The `sample_id` is also used as the simulation ID and as the filename stem of the HBJSON and SQL files.

In [1]:
import warnings
import logging
import shutil
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from tqdm import TqdmWarning, tqdm

warnings.filterwarnings("ignore", message="IProgress not found.*", category=TqdmWarning)

from honeybee.model import Model
from honeybee_energy.run import run_idf
from honeybee_energy.writer import energyplus_idf_version, model_to_idf

from helpers.paths import PROJECT_ROOT, INTERIM_DIR, display_path
from scripts.convert_hbjson_exports_to_idf_epjson import (
    apply_dataset_output_requests_to_idf,
    build_simulation_parameter,
    convert_exports,
)

HBJSONS_DIR = PROJECT_ROOT / "output" / "hbjsons"
SIM_OUTPUT_DIR = (PROJECT_ROOT / "output" / "simulations").resolve()
SIM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SIM_RESULTS_PKL = (INTERIM_DIR / "sim_results_all.pkl").resolve()

# Add a sample ID here to replace an existing result with a new simulation.
FORCE_RERUN_SIM_IDS: set[str] = set()

hbjsons_df = pd.read_pickle(INTERIM_DIR / "hbjsons_df.pkl")
city_weather_df = pd.read_pickle(INTERIM_DIR / "city_weather_assets.pkl")

if hbjsons_df["sample_id"].duplicated().any():
    duplicates = sorted(hbjsons_df.loc[hbjsons_df["sample_id"].duplicated(), "sample_id"].astype(str))
    raise ValueError(f"Duplicate sample IDs in hbjsons_df.pkl: {duplicates[:10]}")

unknown_forced = FORCE_RERUN_SIM_IDS - set(hbjsons_df["sample_id"].astype(str))
if unknown_forced:
    raise KeyError(f"Forced sample IDs are not present in hbjsons_df.pkl: {sorted(unknown_forced)}")

hbjsons_df = hbjsons_df.merge(
    city_weather_df[["city_key", "epw_relpath", "ddy_relpath"]],
    on="city_key",
    how="left",
    validate="many_to_one",
)
missing_weather = hbjsons_df[["epw_relpath", "ddy_relpath"]].isna().any(axis=1)
if missing_weather.any():
    uncovered = sorted(hbjsons_df.loc[missing_weather, "city_key"].astype(str).unique())
    raise KeyError(f"No complete weather assignment for {uncovered}. Re-run BEM4AI_6_Weather.ipynb.")

# Build local paths from the portable filenames written by Stages 5 and 6.
hbjsons_df["file_path"] = hbjsons_df["bhjsons"].map(lambda name: HBJSONS_DIR / str(name))
hbjsons_df["epw_path"] = hbjsons_df["epw_relpath"].map(lambda path: PROJECT_ROOT / str(path))
hbjsons_df["ddy_path"] = hbjsons_df["ddy_relpath"].map(lambda path: PROJECT_ROOT / str(path))

RESULT_COLUMNS = ["idx", "sim_id", "sim_folder", "elapsed", "status", "message"]
if SIM_RESULTS_PKL.is_file():
    sim_results_df = pd.read_pickle(SIM_RESULTS_PKL)
    missing_columns = set(RESULT_COLUMNS) - set(sim_results_df.columns)
    if missing_columns:
        raise ValueError(f"The simulation run log lacks columns: {sorted(missing_columns)}")
else:
    sim_results_df = pd.DataFrame(columns=RESULT_COLUMNS)

successful_ids = set(
    sim_results_df.loc[sim_results_df["status"].eq("success"), "sim_id"].astype(str)
)
sql_ids = {path.stem for path in SIM_OUTPUT_DIR.glob("*.sql") if path.is_file()}
done_sim_ids = (successful_ids & sql_ids) - FORCE_RERUN_SIM_IDS

PENDING_PAIRS = [
    (idx, sim_id)
    for idx, sim_id in hbjsons_df["sample_id"].astype(str).items()
    if sim_id not in done_sim_ids
]
N_TOTAL_MODELS = len(hbjsons_df)
N_PENDING = len(PENDING_PAIRS)
N_DONE_BEFORE = len(done_sim_ids)

forced = f", including {len(FORCE_RERUN_SIM_IDS)} forced" if FORCE_RERUN_SIM_IDS else ""
print(f"{N_TOTAL_MODELS} models across {hbjsons_df['city_key'].nunique()} cities")
print(f"  complete: {N_DONE_BEFORE}")
print(f"  pending:  {N_PENDING}{forced}")
hbjsons_df[["sample_id", "city_key", "epw_relpath", "ddy_relpath"]].head()

12 models across 10 cities
  complete: 0
  pending:  12


,sample_id,city_key,epw_relpath,ddy_relpath
0,Athens_0001,Athens,output/weather/epw/Athens.epw,output/weather/ddy/Athens.ddy
1,Budapest_0001,Budapest,output/weather/epw/Budapest.epw,output/weather/ddy/Budapest.ddy
2,Essen_0001,Essen,output/weather/epw/Essen.epw,output/weather/ddy/Essen.ddy
3,London_0001,London,output/weather/epw/London.epw,output/weather/ddy/London.ddy
4,London_0002,London,output/weather/epw/London.epw,output/weather/ddy/London.ddy


### Convert the reviewed models to IDF and EPJSON

**Inputs**
- Validated and reviewed HBJSON models from Stage 5
- `city_weather_assets.pkl` and the EPW/DDY files produced by Stage 6

**Steps**
1. Translate each HBJSON model with its weather location and DDY design conditions.
2. Apply the EnergyPlus output-variable contract used by the simulation and graph datasets.
3. Validate the resulting IDF and EPJSON files and record the conversion status.

**Outputs**
- Converted models under `output/idf/` and `output/epjson/`
- Conversion summary in `data/interim/hbjson_to_idf_epjson.csv`

Existing converted files are retained unless `OVERWRITE_EXISTING_CONVERSIONS` is enabled.

In [2]:
RUN_IDF_EPJSON_CONVERSION = True
MAX_MODELS_TO_CONVERT = None
OVERWRITE_EXISTING_CONVERSIONS = False
KEEP_CONVERSION_TEMP = False

if not RUN_IDF_EPJSON_CONVERSION:
    print("IDF/EPJSON conversion is disabled.")
    conversion_df = pd.DataFrame()
else:
    conversion_df = convert_exports(
        project_root=PROJECT_ROOT,
        limit=MAX_MODELS_TO_CONVERT,
        overwrite=OVERWRITE_EXISTING_CONVERSIONS,
        keep_temp=KEEP_CONVERSION_TEMP,
    )
    conversion_summary = (
        conversion_df.groupby("status", dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("status")
    )
    display(conversion_summary)

HBJSON -> IDF/EPJSON: 100%|██████████| 12/12 [00:16<00:00,  1.41s/it]


,status,count
0,converted,12


### Maintain the simulation results

The first helper replaces the run-log row for each newly processed model. The second moves a
successful EnergyPlus SQL file out of its temporary run folder and names it by `sample_id`.

In [3]:
def update_run_log(existing: pd.DataFrame, new_rows: list[dict]) -> pd.DataFrame:
    """Return one final run-log row per simulation ID."""
    if not new_rows:
        return existing
    updated = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    return updated.drop_duplicates(subset=["sim_id"], keep="last").reset_index(drop=True)


def store_sql_result(sim_id: str, sql_returned: str | Path) -> Path:
    """Move EnergyPlus output to `output/simulations/<sim_id>.sql`."""
    source = Path(sql_returned)
    if not source.is_file():
        raise FileNotFoundError(f"EnergyPlus returned a missing SQL file: {source}")
    destination = SIM_OUTPUT_DIR / f"{sim_id}.sql"
    source.replace(destination)
    return destination

### Run a batch of pending simulations

**Inputs**
- `hbjsons_df` with model and weather paths, and `PENDING_PAIRS`
- `MAX_MODELS_TO_RUN`, the number of models in this batch, or `None` for all pending models
- `SHOW_EPLUS_LOG`, which controls whether EnergyPlus output is printed

**Steps**
1. Build the simulation parameters and apply the output-variable contract shared with the IDF/EPJSON export utility.
2. Start from a clean per-model run folder, then run EnergyPlus and require a valid SQL result before marking the model as successful.
3. Save final run statuses every ten models so an interrupted batch can resume safely.
4. Delete temporary folders after successful runs and retain failed runs for inspection.

**Outputs**
- One `<sample_id>.sql` for each successful simulation
- `results_this_run` and an updated `data/interim/sim_results_all.pkl`

Failed models remain in `PENDING_PAIRS`, so running this cell again retries them.

In [4]:
# Set to `None` to process every pending model in one batch.
MAX_MODELS_TO_RUN = 1500
SHOW_EPLUS_LOG = False

if MAX_MODELS_TO_RUN is not None and MAX_MODELS_TO_RUN <= 0:
    raise ValueError("MAX_MODELS_TO_RUN must be a positive integer or None.")

if not PENDING_PAIRS:
    print("No pending simulations: every model has a successful run log and an SQL file.")
    results_this_run = []
else:
    to_run_pairs = (
        PENDING_PAIRS
        if MAX_MODELS_TO_RUN is None
        else PENDING_PAIRS[:MAX_MODELS_TO_RUN]
    )
    n_to_run = len(to_run_pairs)
    print(f"Starting simulations for {n_to_run} models (of {N_PENDING} pending)...\n")

    logging.basicConfig(level=logging.INFO)
    results_this_run = []
    results_checkpoint = []

    with tqdm(total=n_to_run, desc="Simulations", position=0, leave=True, miniters=10) as progress:
        for run_index, (idx, sim_id) in enumerate(to_run_pairs, start=1):
            row = hbjsons_df.loc[idx]
            hbjson_path = Path(row["file_path"])
            epw_path = Path(row["epw_path"])
            ddy_path = Path(row["ddy_path"])
            sim_folder = (SIM_OUTPUT_DIR / sim_id).resolve()
            # Remove artifacts from an earlier failed attempt before retrying this model.
            if sim_folder.is_dir():
                shutil.rmtree(sim_folder)
            sim_folder.mkdir(parents=True)

            if SHOW_EPLUS_LOG:
                print(f"\nRunning {sim_id} ({row['city_key']})")
                print(f"HBJSON: {display_path(hbjson_path)}")
                print(f"EPW:    {display_path(epw_path)}")
                print(f"DDY:    {display_path(ddy_path)}")

            start = time.perf_counter()
            status = "exception"
            message = ""

            try:
                for label, path in (("HBJSON", hbjson_path), ("EPW", epw_path), ("DDY", ddy_path)):
                    if not path.is_file():
                        raise FileNotFoundError(f"{label} file not found: {path}")

                model = Model.from_hbjson(str(hbjson_path))
                sim_par = build_simulation_parameter(ddy_path)

                idf_path = sim_folder / "in.idf"
                idf_parts = [sim_par.to_idf(), model_to_idf(model)]
                version_idf = energyplus_idf_version()
                if version_idf:
                    idf_parts.insert(0, version_idf)
                idf_path.write_text("\n\n".join(idf_parts), encoding="utf-8")

                # Use the same explicit variables as the published IDF and EPJSON files.
                apply_dataset_output_requests_to_idf(idf_path)

                sql, _zsz, _rdd, _html, error_file = run_idf(
                    idf_file_path=str(idf_path),
                    epw_file_path=str(epw_path),
                    expand_objects=True,
                    silent=not SHOW_EPLUS_LOG,
                )

                failed_job = sim_folder / "run" / "failed.job"
                if failed_job.is_file():
                    status = "failed_job"
                    message = f"EnergyPlus reported a failed job; inspect {display_path(sim_folder)}."
                elif not sql or not Path(sql).is_file():
                    status = "failed_no_sql"
                    detail = f" Error file: {error_file}" if error_file else ""
                    message = f"EnergyPlus did not produce an SQL file.{detail}"
                else:
                    stored_sql = store_sql_result(sim_id, sql)
                    status = "success"
                    message = f"SQL saved to {display_path(stored_sql)}."

            except Exception as exc:  # Continue the batch and record this model's failure.
                message = str(exc)
                logging.exception("Simulation failed for %s", sim_id)

            elapsed = time.perf_counter() - start

            # A failed forced or replacement run must not leave an older SQL in the result set.
            if status != "success":
                stale_sql = SIM_OUTPUT_DIR / f"{sim_id}.sql"
                if stale_sql.is_file():
                    stale_sql.unlink()
            elif sim_folder.is_dir():
                shutil.rmtree(sim_folder, ignore_errors=True)

            result = {
                "idx": idx,
                "sim_id": sim_id,
                "sim_folder": display_path(sim_folder),
                "elapsed": elapsed,
                "status": status,
                "message": message,
            }
            results_this_run.append(result)
            results_checkpoint.append(result)

            if run_index % 10 == 0 or run_index == n_to_run:
                sim_results_df = update_run_log(sim_results_df, results_checkpoint)
                sim_results_df.to_pickle(SIM_RESULTS_PKL)
                results_checkpoint = []
                if SHOW_EPLUS_LOG:
                    print(f"Checkpoint saved with {len(sim_results_df)} run-log rows.")

            progress.update(1)

    batch_results = pd.DataFrame(results_this_run)
    print("\nSimulation status counts for this batch:")
    print(batch_results["status"].value_counts().to_string())
    display(batch_results.head(20))

    completed_this_batch = set(
        batch_results.loc[batch_results["status"].eq("success"), "sim_id"].astype(str)
    )
    PENDING_PAIRS = [pair for pair in PENDING_PAIRS if pair[1] not in completed_this_batch]
    N_PENDING = len(PENDING_PAIRS)
    print(f"\nPending simulations remaining after this batch: {N_PENDING}")

Starting simulations for 12 models (of 12 pending)...



Simulations:   0%|          | 0/12 [00:00<?, ?it/s]

ExpandObjects Started.
 Begin reading Energy+.idd file.
 Done reading Energy+.idd file.
ExpandObjects Finished. Time:     0.041
EnergyPlus Starting
EnergyPlus, Version 25.1.0-1c11a3d85f, YMD=2026.09.14 07:58
Initializing Response Factors
Calculating CTFs for "GENERIC INTERIOR FLOOR"
Calculating CTFs for "BS ROOF GREECE TRADE 1945 - 1969"
Calculating CTFs for "BS WALL GREECE TRADE 1945 - 1969"
Calculating CTFs for "BS FLOOR GREECE TRADE 1945 - 1969"
Initializing Window Optical Properties
Initializing Solar Calculations
Allocate Solar Module Arrays
Initializing Zone and Enclosure Report Variables
Initializing Surface (Shading) Report Variables
Computing Interior Solar Absorption Factors
Determining Shadowing Combinations
Computing Window Shade Absorption Factors
Initializing Solar Reflection Factors
Proceeding with Initializing Solar Calculations
Calculating Beam-to-Diffuse Exterior Solar Reflection Factors
Calculating Beam-to-Beam Exterior Solar Reflection Factors
Calculating Sky Diffus

Simulations:   0%|          | 0/12 [00:12<?, ?it/s]

Updating Beam-to-Beam Exterior Solar Reflection Factors
Warming up {2}
Warming up {3}
Warming up {4}
Warming up {5}
Starting Simulation at 09/21 for ATHINAI-HELLINIKON.OLYMPIC.CO SEPTEMBER .4% CONDNS WB=>MCDB
Initializing New Environment Parameters
Warming up {1}
Updating Beam-to-Diffuse Exterior Solar Reflection Factors
Updating Beam-to-Beam Exterior Solar Reflection Factors
Warming up {2}
Warming up {3}
Warming up {4}
Starting Simulation at 10/21 for ATHINAI-HELLINIKON.OLYMPIC.CO OCTOBER .4% CONDNS WB=>MCDB
Initializing New Environment Parameters
Warming up {1}
Updating Beam-to-Diffuse Exterior Solar Reflection Factors
Updating Beam-to-Beam Exterior Solar Reflection Factors
Warming up {2}
Warming up {3}
Warming up {4}
Starting Simulation at 11/21 for ATHINAI-HELLINIKON.OLYMPIC.CO NOVEMBER .4% CONDNS WB=>MCDB
Initializing New Environment Parameters
Warming up {1}
Updating Beam-to-Diffuse Exterior Solar Reflection Factors
Updating Beam-to-Beam Exterior Solar Reflection Factors
Warming 

...

Writing tabular output file results using comma format.
Writing final SQL reports
EnergyPlus Run Time=00hr 00min 46.74sec
ExpandObjects Started.
 Begin reading Energy+.idd file.
 Done reading Energy+.idd file.
ExpandObjects Finished. Time:     0.060
EnergyPlus Starting
EnergyPlus, Version 25.1.0-1c11a3d85f, YMD=2026.09.14 08:05
Initializing Response Factors
Calculating CTFs for "BS FLOOR POLAND OFFICES 1945 - 1969"
Calculating CTFs for "BS ROOF POLAND OFFICES 1945 - 1969"
Calculating CTFs for "GENERIC INTERIOR FLOOR"
Calculating CTFs for "BS WALL POLAND OFFICES 1945 - 1969"
Calculating CTFs for "GENERIC INTERIOR WALL"
Initializing Window Optical Properties
Initializing Solar Calculations
Allocate Solar Module Arrays
Initializing Zone and Enclosure Report Variables
Initializing Surface (Shading) Report Variables
Computing Interior Solar Absorption Factors
Determining Shadowing Combinations
Computing Window Shade Absorption Factors
Initializing Solar Reflection Factors
Proceeding with In

EnergyPlus Completed Successfully.
Simulations: 100%|██████████| 12/12 [08:07<00:00, 40.67s/it]


Simulation status counts for this batch:
status
success    12


,idx,sim_id,sim_folder,elapsed,status,message
0,0,Athens_0001,outp...,32.274579,success,SQL saved to output/simulations/Athens_0001.sql.
1,1,Budapest_0001,outp...,28.190699,success,SQL saved to output/simulations/Budapest_0001....
2,2,Essen_0001,outp...,80.359254,success,SQL saved to output/simulations/Essen_0001.sql.
3,3,London_0001,outp...,12.965484,success,SQL saved to output/simulations/London_0001.sql.
4,4,London_0002,outp...,17.163870,success,SQL saved to output/simulations/London_0002.sql.
5,5,London_0003,outp...,79.280758,success,SQL saved to output/simulations/London_0003.sql.
6,6,Madrid_0002,outp...,5.487349,success,SQL saved to output/simulations/Madrid_0002.sql.
7,7,Manchester_0001,outp...,20.911697,success,SQL saved to output/simulations/Manchester_000...
8,8,Milan_0001,outp...,42.497161,success,SQL saved to output/simulations/Milan_0001.sql.
9,9,Rome_0001,outp...,5.477614,success,SQL saved to output/simulations/Rome_0001.sql.



Pending simulations remaining after this batch: 0


### Summarize simulation coverage

Compare successful run-log rows with the SQL files on disk and report any models that still need
to be simulated. When a batch ran above, also display its final statuses and messages.

In [5]:
sim_results_df = (
    pd.read_pickle(SIM_RESULTS_PKL) if SIM_RESULTS_PKL.is_file() else sim_results_df
)
successful_ids = set(
    sim_results_df.loc[sim_results_df["status"].eq("success"), "sim_id"].astype(str)
)
sql_ids = {path.stem for path in SIM_OUTPUT_DIR.glob("*.sql") if path.is_file()}
completed_ids = successful_ids & sql_ids
all_model_ids = set(hbjsons_df["sample_id"].astype(str))
still_missing = sorted(all_model_ids - completed_ids)

completed_this_batch = sum(
    result["status"] == "success" for result in results_this_run
) if "results_this_run" in globals() else 0

print(f"Total models:             {N_TOTAL_MODELS}")
print(f"  complete before batch:  {N_DONE_BEFORE}")
print(f"  completed in batch:     {completed_this_batch}")
print(f"  complete in total:      {len(all_model_ids & completed_ids)}")
print(f"  still pending:          {len(still_missing)}")
if still_missing:
    print("  " + ", ".join(still_missing[:10]))

if results_this_run:
    display(pd.DataFrame(results_this_run))

Total models:             12
  complete before batch:  0
  completed in batch:     12
  complete in total:      12
  still pending:          0


,idx,sim_id,sim_folder,elapsed,status,message
0,0,Athens_0001,outp...,32.274579,success,SQL saved to output/simulations/Athens_0001.sql.
1,1,Budapest_0001,outp...,28.190699,success,SQL saved to output/simulations/Budapest_0001....
2,2,Essen_0001,outp...,80.359254,success,SQL saved to output/simulations/Essen_0001.sql.
3,3,London_0001,outp...,12.965484,success,SQL saved to output/simulations/London_0001.sql.
4,4,London_0002,outp...,17.163870,success,SQL saved to output/simulations/London_0002.sql.
5,5,London_0003,outp...,79.280758,success,SQL saved to output/simulations/London_0003.sql.
6,6,Madrid_0002,outp...,5.487349,success,SQL saved to output/simulations/Madrid_0002.sql.
7,7,Manchester_0001,outp...,20.911697,success,SQL saved to output/simulations/Manchester_000...
8,8,Milan_0001,outp...,42.497161,success,SQL saved to output/simulations/Milan_0001.sql.
9,9,Rome_0001,outp...,5.477614,success,SQL saved to output/simulations/Rome_0001.sql.
